# OmniVoice Zero-Shot Voice Cloning

This notebook runs zero-shot voice cloning with `k2-fsa/OmniVoice` from a reference audio clip. It does not fine-tune or train weights; it uses `ref_audio` and `ref_text` at inference time.

## 1. Runtime

This notebook is configured for Apple Silicon Mac GPU via PyTorch MPS. Run it from the repo root or set `ref_audio` to an absolute local path.

In [3]:
import torch

assert torch.backends.mps.is_available(), "PyTorch MPS is not available on this Mac."
print("MPS available:", torch.backends.mps.is_available())
print("PyTorch:", torch.__version__)

MPS available: True
PyTorch: 2.10.0


## 2. Install Dependencies

In [5]:
!pip install -U pip
!pip install torch==2.8.0 torchaudio==2.8.0 omnivoice soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 8.5 MB/s  0:00:08m0:00:0100:01
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0
    Uninstalling torch-2.10.0:
      Successfully uninstalled torch-2.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0 requires torch==2.10.0, but you have torch 2.8.0 which is incompatible.


## 3. Set Reference Audio

Use a clean WAV file, ideally 5-15 seconds long, with one speaker and minimal background noise. The `ref_text` should be the exact transcript of the reference audio.

In [8]:
from pathlib import Path

ref_audio = "sin_2282_8643512444.wav"

if not Path(ref_audio).exists():
    raise FileNotFoundError(
        f"Could not find {ref_audio!r}. Put your reference WAV beside this notebook "
        "or set ref_audio to an absolute path."
    )

print("Reference audio:", ref_audio)

Reference audio: sin_2282_8643512444.wav


In [17]:
ref_text = "සම්පත් බැංකුවේ ආරම්භක මුදල අවම නමුත් පවත්වාගැනීමේ වාර්ෂික මුදල ඉහළ නිසා මුල් ලාභය ඉන් නැතිවී යනවා"

target_text = "මෙම නිදහස් අරගලයෙහි නායකයන් වූයේ විල්බාවේ සහ කැප්පෙටිපොල දිසාවේය"
output_wav = "out.wav"

## 4. Load OmniVoice

In [14]:
from omnivoice import OmniVoice
import torch

device = "mps"

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=torch.float16,
)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

## 5. Generate Audio

In [18]:
import soundfile as sf

audio = model.generate(
    text=target_text,
    ref_audio=ref_audio,
    ref_text=ref_text,
)

sf.write(output_wav, audio[0], 24000)
print("Wrote", output_wav)

Wrote out.wav


## 6. Listen and Download

In [19]:
from IPython.display import Audio, display

display(Audio(output_wav, rate=24000))

In [ ]:
print("Saved audio to", output_wav)

## Optional: Use a Repo Sample

If you are running from this repo root, you can test with an existing sample.

In [ ]:
# ref_audio = "homelands_omnivoice_sample.wav"
# ref_text = "Replace this with the exact transcript of the sample audio."